# Known-k Temporal-GCN: независимая проверка
Три уже обученные модели сравниваются с фиксированным known-k Temporal-v3 на 1998 новых IC-каскадах той же Facebook-сети. Snapshot S1b с неизвестным числом источников показывается отдельно. Обучения и настройки здесь нет. Выполняй только указанные ячейки вручную; защитные STOP-ячейки предназначены для остановки «Выполнить всё» и вручную не нужны.

In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys
if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Этот notebook запускается в Google Colab')
if any(name == 'diffusion_sources' or name.startswith('diffusion_sources.') or name.startswith('scripts.known_k_independent') for name in sys.modules):
    raise RuntimeError('Код уже импортирован: перезапусти среду Colab перед setup')
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/diffusion-sources-known-k-independent')
REMOTE = 'https://github.com/1habibi/diffusion-sources-localization-.git'
BRANCH = 'experiment/known-k-temporal-gcn-independent'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(PROJECT)], check=True)
else:
    if not (PROJECT / '.git').exists():
        raise RuntimeError(f'{PROJECT} существует, но не является Git-клоном')
    clean = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()
    branch = subprocess.check_output(['git', '-C', str(PROJECT), 'branch', '--show-current'], text=True).strip()
    if clean or branch != BRANCH:
        raise RuntimeError('Клон изменён или открыта другая ветка; ничего не перезаписываем')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
packages = {'numpy': 'numpy', 'networkx': 'networkx', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML', 'tqdm': 'tqdm', 'torch_geometric': 'torch-geometric'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT), '--no-deps'], check=True)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
import torch
from scripts import known_k_independent_artifacts as INDEPENDENT
from scripts.known_k_independent_artifacts import IndependentKnownKPaths, preflight, freeze_inputs
from scripts.known_k_independent_generation import generate_and_seal
from scripts.known_k_independent_inference import open_evaluation, evaluate_seed
from scripts.known_k_independent_summary import save_summary
if not Path(INDEPENDENT.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Kernel импортировал другую копию проекта; перезапусти среду')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
REVISION = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
print('Project:', PROJECT, 'revision:', REVISION, 'device:', DEVICE)
print('Setup только установил код. Freeze, seal, open и inference не выполнялись.')


In [ ]:
ROOT = globals().get('ROOT_OVERRIDE', Path('/content/drive/MyDrive/diffusion-sources'))
PATHS = IndependentKnownKPaths(
    repo=PROJECT, reference=ROOT / 'data/facebook_main',
    snapshot_holdout=ROOT / 'data/facebook_snapshot_final_holdout',
    prior_independent=ROOT / 'data/generated/facebook_temporal_v3_independent_holdout',
    pilot=ROOT / 'reports/runs/known_k_temporal_gcn_pilot/v1',
    repeats=ROOT / 'reports/runs/known_k_temporal_gcn_repeats/v1',
    s1b=ROOT / 'reports/runs/facebook_main_v2/s1b/seed_7026',
    data=ROOT / 'data/generated/facebook_known_k_temporal_gcn_independent_v1',
    reports=ROOT / 'reports/runs/known_k_temporal_gcn_independent_evaluation/v1',
    generation_config=PROJECT / 'configs/facebook_known_k_temporal_gcn_independent_holdout.yaml',
    notebook=PROJECT / 'notebooks/colab_known_k_temporal_gcn_independent.ipynb',
)
required = [PATHS.reference / 'graph.npz', PATHS.reference / 'train.npz',
            PATHS.reference / 'validation.npz', PATHS.reference / 'test.npz',
            PATHS.snapshot_holdout / 'final_holdout.npz',
            PATHS.prior_independent / 'independent_holdout.npz',
            PATHS.pilot / 'pilot/complete', PATHS.repeats / 'summary/complete',
            PATHS.s1b / 'best_model.pt', PATHS.generation_config, PATHS.notebook]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)
raw = PROJECT / 'data/raw/facebook_combined.txt.gz'
if not raw.is_file():
    import urllib.request
    raw.parent.mkdir(parents=True, exist_ok=True)
    print('Provisioning raw graph only:', 'https://snap.stanford.edu/data/facebook_combined.txt.gz')
    urllib.request.urlretrieve('https://snap.stanford.edu/data/facebook_combined.txt.gz', raw)
print('Reference:', PATHS.reference, 'snapshot metadata:', PATHS.snapshot_holdout)
print('New data:', PATHS.data, 'new reports:', PATHS.reports)
print('Paths checked; next cell is read-only preflight.')


In [ ]:
CHECK = preflight(PATHS)
identity = CHECK['identity']
print('Authenticated checkpoints:', identity['checkpoints'])
print('Old summary:', CHECK['summary']['status'], 'old gate:', CHECK['summary']['positive_delta_each_repeat'])
print('Dataset seed:', identity['protocol']['dataset_seed'], 'n:', identity['protocol']['n'])
print('Preflight did not create or open a dataset. Next cell is freeze.')


In [ ]:
FROZEN = freeze_inputs(PATHS)
print('Freeze:', FROZEN['status'], 'dataset seed:', FROZEN['identity']['protocol']['dataset_seed'])
print('Manifest:', PATHS.reports / 'freeze/manifest.json')
print('STOP: пришли вывод freeze. Генерация ещё не запущена.')


In [ ]:
raise RuntimeError('STOP после freeze: не запускай всё подряд; после согласования вручную выбери ячейку seal')


In [ ]:
SEALED = generate_and_seal(PATHS)
print('Seal:', {key: SEALED[key] for key in ('evaluation_status', 'n', 'dataset_seed', 'target_metrics_computed', 'seed_isolation')})
print('Manifest:', PATHS.reports / 'seal/manifest.json')
print('STOP: пришли seal. Набор закрыт; ячейку open не запускай без отдельного согласования.')


In [ ]:
raise RuntimeError('STOP перед open: требуется отдельное согласование после seal; вручную пропусти эту защитную ячейку')


In [ ]:
answer = input('Для независимой оценки введи OPEN_KNOWN_K_INDEPENDENT_HOLDOUT: ').strip()
OPENED = open_evaluation(PATHS, answer)
print('Opened:', OPENED['evaluation_status'], 'marker:', PATHS.reports / 'opened/manifest.json')
print('Теперь можно по одной запускать три ячейки frozen inference; обучения нет.')


In [ ]:
REPORT_7026 = evaluate_seed(PATHS, 7026, torch.device(DEVICE))
print('Seed 7026:', {key: REPORT_7026[key] for key in ('delta_f1', 'f1_ci')})
print('Control F1:', REPORT_7026['control']['all']['f1'], 'GCN F1:', REPORT_7026['candidate']['all']['f1'])
print('STOP: пришли короткий вывод 7026; 7027 не запускай автоматически.')


In [ ]:
raise RuntimeError('STOP после seed 7026: после проверки вручную выбери ячейку seed_7027')


In [ ]:
REPORT_7027 = evaluate_seed(PATHS, 7027, torch.device(DEVICE))
print('Seed 7027:', {key: REPORT_7027[key] for key in ('delta_f1', 'f1_ci')})
print('Control F1:', REPORT_7027['control']['all']['f1'], 'GCN F1:', REPORT_7027['candidate']['all']['f1'])
print('STOP: пришли короткий вывод 7027; 7028 не запускай автоматически.')


In [ ]:
raise RuntimeError('STOP после seed 7027: после проверки вручную выбери ячейку seed_7028')


In [ ]:
REPORT_7028 = evaluate_seed(PATHS, 7028, torch.device(DEVICE))
print('Seed 7028:', {key: REPORT_7028[key] for key in ('delta_f1', 'f1_ci')})
print('Control F1:', REPORT_7028['control']['all']['f1'], 'GCN F1:', REPORT_7028['candidate']['all']['f1'])
print('STOP: пришли короткий вывод 7028; summary после проверки.')


In [ ]:
raise RuntimeError('STOP после seed 7028: после проверки вручную выбери ячейку summary')


In [ ]:
SUMMARY = save_summary(PATHS)
print('Primary:', SUMMARY['primary'])
print('Mean:', SUMMARY['mean'], 'sample SD:', SUMMARY['sample_sd'])
print('Snapshot estimated-k:', SUMMARY['snapshot_estimated']['mean'])
print('Summary:', PATHS.reports / 'summary/payload.json')
print('STOP: пришли summary. Это новые каскады той же топологии, не перенос на другой граф.')
